In [20]:
import pandas as pd
df=pd.read_csv('data/netflix_cleaned.csv')
print(df.shape)
df.head()

(8790, 19)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,duration_int,duration_minutes,duration_seasons,primary_country,primary_genre
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021,9,90.0,90.0,NaN,United States,Documentaries
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021,9,2.0,NaN,2.0,South Africa,International TV Shows
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021,9,1.0,NaN,1.0,Unknown,Crime TV Shows
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021,9,1.0,NaN,1.0,Unknown,Docuseries
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021,9,2.0,NaN,2.0,India,International TV Shows


In [21]:
#filling remaining NaN again for safety
for col in ['description','director','listed_in','cast']:
    df[col]=df[col].fillna('')

df['combined_features']=(
    df['listed_in']+' '+
    df['description']+' '+
    df['director']+' '+
    df['cast']
)

df['combined_features'].iloc[0]

'Documentaries As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable. Kirsten Johnson Unknown'

In [22]:
print(df[['title','combined_features']].sample(3))

                     title                                  combined_features
177   The Guns of Navarone  Action & Adventure, Classic Movies During Worl...
4415            Brainchild  Kids' TV From germs and emotions to social med...
5982          12 ROUND GUN  Dramas, Independent Movies, Sports Movies Deal...


In [23]:
#Converting combined text into numeric vectors using TF-IDF
# TF-IDF turns each title's text into a vector of word importance scores, so titles with similar words end up with similar vectors.
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(stop_words='english')
tfidf_matrix=tfidf.fit_transform(df['combined_features'])
tfidf_matrix.shape

(8790, 49895)

In [24]:
#Computing cosine similarity between every pair of titles
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim=cosine_similarity(tfidf_matrix,tfidf_matrix)
cosine_sim.shape


(8790, 8790)

In [25]:
#Build a lookup so we can go from title name → row index
indices=pd.Series(df.index, index=df['title']).drop_duplicates()
indices['Squid Game']

np.int64(33)

In [26]:
#Building recommendation function
def get_recommendations(title, cosine_sim=cosine_sim, df=df, indices=indices):
    if title not in indices:
        return f"{title} not in dataset. Check spelling"
    idx=indices[title]

    #if there are duplicate titles
    if isinstance(idx, pd.Series):
        idx=idx.iloc[0]

    #getting similarity scores
    sim_scores=list(enumerate(cosine_sim[idx]))

    sim_scores=sorted(sim_scores, key=lambda x:x[1], reverse=True)
    sim_scores=sim_scores[1:6]

    title_indices=[i[0] for i in sim_scores]
    return df[['title','type','primary_genre','release_year']].iloc[title_indices]

In [27]:
get_recommendations('Alice in Borderland')

,title,type,primary_genre,release_year
1965,High & Low The Movie 2 / End of Sky,Movie,Action & Adventure,2017
1969,Road To High & Low,Movie,Action & Adventure,2016
2875,Followers,TV Show,International TV Shows,2020
2066,Erased,TV Show,Anime Series,2016
1966,High & Low The Movie 3 / Final Mission,Movie,Action & Adventure,2017


In [28]:
import pickle

df[['title', 'type', 'primary_genre', 'release_year', 'description']].to_csv('netflix_titles.csv', index=False)

with open('cosine_sim.pkl','wb') as f:
    pickle.dump(cosine_sim, f)
with open('indices.pkl', 'wb') as f:
    pickle.dump(indices, f)

In [29]:
import json

indices_dict = indices.to_dict()
with open('indices.json', 'w') as f:
    json.dump(indices_dict, f)